In [14]:
import os
from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from email.message import EmailMessage
import smtplib
from quiet import silence
import board
silence()  # turn down some noisy library logging so the agent's trace is easy to read

os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "FALSE") # make sure we dont use vertext ai to talk to gemini
load_dotenv(override=True)

True

In [4]:
# Create an agent
MODEL = "gemini-3.1-flash-lite"
agent = LlmAgent(
    model=MODEL,
    name="assistant",
    instruction="You are a concise, friendly assistant. Reply in a single short sentence."
)

In [13]:
# Run the agent
result = await InMemoryRunner(agent=agent).run_debug("Say hello in Spanish.", verbose=True)

assistant > ¡Hola!


In [3]:
board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt',
  'status': 'pending',
  'result': ''}]

In [4]:
board.show_board()

Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt

In [2]:
# Add tools
# A tool in ADK is just a typed Python function with a docstring. The docstring becomes the description the model reads, the type hints
# become the argument schema, and you hand the function to the agent in its tools list

def show_todos() -> list[dict]:
    """List every todo on the board. A goal has a parent_id None; a step has a parent_id set to it's goal's id"""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions"""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and a record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}


In [5]:
board_agent = LlmAgent(
    model= MODEL,
    name="board_agent",
    instruction="You help manage a shared todo board.",
    tools=[show_todos, complete_task]
)

In [6]:
result = await InMemoryRunner(agent=board_agent).run_debug("What is on the board right now, and what is its status?", verbose=True)

board_agent > [Calling tool: show_todos({})]
board_agent > [Tool result: {'result': [{'id': 1, 'parent_id': None, 'task': 'Read notes.txt, translate its contents into natura...]
board_agent > On the board, there is one goal pending:

*   **Task:** "Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt"
*   **Status:** Pending


In [15]:

EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

def send_email(subject, text_body, html_body):
    """
    Send out an email with the given subject and body
    Args:
     subject: The subject of the email
     text_body: The body of the email as plain text
     html_body: The HTML of the email 
    """
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)


In [16]:
# Add MCP
# MCP is just more tools: ones you did not write, connected over a small protocal. We give the agent the filesystem reference server
# scoped to a single workspace folder so it can only touch files in there.In ADK that is one McpToolset added to the same tools=[] list
# ADK starts the server for you as a small Node process over npx. One detail worth knowing: we pass errlog=subprocess.DEVNULL. That
# quiets the server's startup logging

import subprocess
from pathlib import Path
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from mcp import StdioServerParameters

workspace = Path("task_worker/workspace").resolve() # the only folder the agent may touch

filesystem = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
            cwd=str(workspace),  # start the server in the workspace so relative file names resolve there
        ),
        timeout=60
    ),
    errlog=subprocess.DEVNULL
)

In [17]:
file_agent = LlmAgent(
    model=MODEL,
    name="file_agent",
    instruction="You can read and write files in your workspace. Use your tools to do what is asked",
    tools=[filesystem]
)

In [10]:
result = await InMemoryRunner(agent=file_agent).run_debug("Read notes.txt and summarize it in one sentence", verbose=True)

file_agent > [Calling tool: list_allowed_directories({})]
file_agent > [Tool result: {'content': [{'type': 'text', 'text': 'Allowed directories:\n/Users/denniskabuye/Desktop/AI/Ai-Proje...]
file_agent > [Calling tool: list_directory({'path': '.'})]
file_agent > [Tool result: {'content': [{'type': 'text', 'text': '[FILE] notes.txt'}], 'structuredContent': {'content': '[FILE]...]
file_agent > [Calling tool: read_text_file({'path': 'notes.txt'})]
file_agent > [Tool result: {'content': [{'type': 'text', 'text': 'Welcome to the team.\n\nToday we are building a small languag...]
file_agent > The team is collaborating on building a small language tutor by having members carefully complete individual tasks assigned from a shared board.


In [18]:
# Put it in a loop with a goal
# Now the payoff. Give one agent all three board tools and the filesystem server, hand it a goal and let it run. It plans its own steps
# on the board, works them with its file tools, ticks each one off, and closes the goal when the work is done. That is the agent loop 
# working on its own: read, plan, act, checkoff, repeat. This time we drive the loop with run_async rather than run_debug, stepping 
# through its events ourselves. We print each tool call as the agent makes it, and re-draw the board whenever a step is planned or 
# ticked off, so you watch the plan appear and then strike through one line at a time

from google.genai import types

INSTRUCTIONS="""
You are a careful worker with a shared todo board and a set of file tools.
Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down
into, added to the board under the goal. Then carry them out with your file tools, making each step done as you finish it. Once the
steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = LlmAgent(
    model=MODEL,
    name="task_worker",
    instruction=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem, send_email]
)
board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt and send a email with a summary.")
board.claim_todo(goal_id)

# Drive the loop ourselves with run_async instead of run_debug, so we can watch it:
# print each tool call, and re-draw the board whenever a step is planned or ticked off

runner = InMemoryRunner(agent=worker)
session = await runner.session_service.create_session(app_name=runner.app_name, user_id="denno")

async for event in runner.run_async(
    user_id="denno",
    session_id=session.id,
    new_message=types.UserContent("Please work the pending goal on the board."),
):
    for call in event.get_function_calls():
        print(f"tool: {call.name}({call.args})")
        if any(r.name in ("plan_steps", "complete_task") for r in event.get_function_responses()):
            board.show_board()
        if event.is_final_response():
            print(event.content.parts[0].text)


tool: show_todos({})
tool: list_allowed_directories({})
tool: list_directory({'path': '/Users/denniskabuye/Desktop/AI/Ai-Projects/other_framewors/task_worker/workspace'})
tool: plan_steps({'steps': ['Read the content of notes.txt.', 'Translate the content into natural Spanish.', 'Write the Spanish text to spanish.txt.', 'Send an email with the summary of the work.'], 'goal_id': 1})
tool: read_text_file({'path': 'notes.txt'})
tool: complete_task({'task_id': 2, 'result': 'Read the content of notes.txt.'})
tool: complete_task({'result': 'Translated the text into natural Spanish.', 'task_id': 3})
tool: write_file({'path': 'spanish.txt', 'content': 'Bienvenido al equipo.\n\nHoy estamos construyendo un pequeño tutor de idiomas, una pieza a la vez. Cada asistente elige una sola tarea del tablero compartido, hace el trabajo con cuidado y marca la tarea como terminada. Lee tu tarea detenidamente, tómate tu tiempo y haz tu mejor esfuerzo.\n'})
tool: complete_task({'result': 'Wrote the translated